<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Traffic_near_miss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip -q install ultralytics supervision opencv-python-headless numpy

In [9]:
!pip -q install fiftyone ultralytics supervision opencv-python-headless

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.6/112.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.1/313.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 24.6 MB/s eta 0:0

In [10]:
import fiftyone as fo
import fiftyone.zoo as foz

# Download a sample driving video (rainy / low visibility included)
dataset = foz.load_zoo_dataset(
    "quickstart-video",
    max_samples=1
)

sample = dataset.first()
video_path = sample.filepath

video_path

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


INFO:fiftyone.zoo.datasets:Downloading dataset to '/root/fiftyone/quickstart-video'


INFO:fiftyone.zoo.datasets.base:Downloading dataset...


 100% |████|  281.7Mb/281.7Mb [339.8ms elapsed, 0s remaining, 828.8Mb/s]      


INFO:eta.core.utils: 100% |████|  281.7Mb/281.7Mb [339.8ms elapsed, 0s remaining, 828.8Mb/s]      


...

 100% |█████████████████████| 1/1 [716.4ms elapsed, 0s remaining, 1.4 samples/s] 


INFO:eta.core.utils: 100% |█████████████████████| 1/1 [716.4ms elapsed, 0s remaining, 1.4 samples/s] 


Dataset 'quickstart-video-1' created


INFO:fiftyone.zoo.datasets:Dataset 'quickstart-video-1' created


'/root/fiftyone/quickstart-video/data/Ulcb3AjxM5g_053-1.mp4'

In [11]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/driving_video
!cp "{video_path}" /content/drive/MyDrive/driving_video/rain_video.mp4

Mounted at /content/drive


In [12]:
VIDEO_IN  = "/content/drive/MyDrive/driving_video/rain_video.mp4"
VIDEO_OUT = "/content/drive/MyDrive/driving_video/rain_near_miss_output.mp4"

In [14]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv

# --- Model ---
model = YOLO("yolov8n.pt")  # or yolov8s.pt for better results

# COCO class IDs we care about:
# 0=person, 1=bicycle, 2=car, 3=motorcycle, 5=bus, 7=truck
KEEP_CLASS_IDS = {0, 1, 2, 3, 5, 7}

# --- Video IO ---
cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_IN}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (w, h))

# --- Tracker + Annotators ---
tracker = sv.ByteTrack(frame_rate=fps)

box_annotator   = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

print("FPS:", fps, "Size:", (w, h))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FPS: 29.97002997002997 Size: (1920, 1080)


In [15]:
from collections import defaultdict, deque

In [16]:
traj = defaultdict(lambda: deque(maxlen=25))  # last N centers per track

In [18]:
for i in range(len(tracked)):
    x1, y1, x2, y2 = tracked.xyxy[i]

    track_id = int(tracked.tracker_id[i]) if tracked.tracker_id is not None else -1
    cls_id   = int(tracked.class_id[i]) if tracked.class_id is not None else -1
    conf     = float(tracked.confidence[i]) if tracked.confidence is not None else 0.0

    cx = float((x1 + x2) / 2.0)
    cy = float((y1 + y2) / 2.0)
    area = float((x2 - x1) * (y2 - y1))

    # ✅ trajectory update MUST be here
    traj[track_id].append((int(cx), int(cy)))

    is_nm, score, details = compute_near_miss(
        track_id, cx, cy, area, fps
    )

    name = model.names.get(cls_id, str(cls_id))
    base = f"ID {track_id} {name} {conf:.2f}"

    if is_nm:
        base += f"  !!! NM score={score:.2f}"

    labels.append(base)

NameError: name 'tracked' is not defined

In [17]:
traj[track_id].append((int(cx), int(cy)))

NameError: name 'track_id' is not defined